In [0]:
import pyspark.sql.functions as F
from pyspark.sql import DataFrame
from pyspark.sql.window import Window # 如果需要复杂的去重逻辑备用

class NYC_Taxi_Silver_Loader:
    def __init__(self, spark, run_id, target_table: str):
        self.spark = spark
        self.target_table = target_table
        self.quarantine_table = f"{target_table}_quarantine"
        self.run_id = run_id
        
        # 优化：将业务规则封装，便于后续作为配置文件(如 JSON/YAML)动态加载
        self.BASE_RULES = {
            "missing_pickup": F.col("pickup_datetime").isNull(),
            "missing_dropoff": F.col("dropoff_datetime").isNull(),
            "dropoff_before_pickup": (F.col("pickup_datetime").isNotNull()) & 
                                     (F.col("dropoff_datetime").isNotNull()) & 
                                     (F.col("dropoff_datetime") < F.col("pickup_datetime")),
            "passenger_count_invalid": (F.col("passenger_count") < 0) | (F.col("passenger_count") > 9),
            "total_amount_negative": F.col("total_amount") < 0,
            "invalid_YYYYMM": (F.col("YYYYMM") < 190001) | (F.col("YYYYMM") > 300012),
            "duration_out_of_range": (F.col("duration_min") < 2.0) | (F.col("duration_min") > 180),
            "distance_too_small": F.col("trip_distance") <= 0.1,
            "efficiency_too_high": F.col("temp_eff") >= 15.0,
            "fare_too_low": F.col("fare_amount") <= 2.5
        }

    def process(self, bronze_df: DataFrame) -> None:
        """
        主处理流程：特征提取 -> 规则校验 -> 持久化缓存 -> 数据分流写入
        """
        
        # --- 1. 特征提取与 Audit (审计) 元数据注入 ---
        # 最佳实践：使用 select 替代多个连续的 withColumn 以优化 Catalyst 执行计划
        
        #enriched_df = bronze_df.withColumnRenamed("_run_id", "brz_run_id").withColumnRenamed#("_processed_at", "bnz_processed_at").select(
       #     "*",
       #     ((F.col("dropoff_datetime").cast("long") - F.col("pickup_datetime").cast#("long")) / 60.0).alias("duration_min"),
            # 注入数据血缘和审计字段
      #      F.lit(self.run_id).alias("_run_id"),
      #      F.current_timestamp().alias("_processed_at")
        #).withColumn(
        #    "temp_eff", 
        #    F.when(F.col("duration_min") > 0, F.col("fare_amount") / F.col("duration_min")).#otherwise(F.lit(0))
        #)

        enriched_df = bronze_df.withColumnRenamed("_run_id", "brz_run_id").withColumnRenamed("_processed_at", "bnz_processed_at").select(
            "*",
            ((F.col("dropoff_datetime").cast("long") -
            F.col("pickup_datetime").cast("long")) / 60.0).alias("duration_min"),
            F.lit(self.run_id).alias("_run_id"),
            F.current_timestamp().alias("_processed_at")
        )

        enriched_df = enriched_df.withColumn(
            "temp_eff",
            F.when(
                F.col("duration_min") > 0,
                F.col("fare_amount") / F.col("duration_min")
            ).otherwise(F.lit(0.0))
        )
    

        # --- 2. 内存级规则校验 (避免 Schema 污染) ---
        # 最佳实践：直接在数组中进行条件判断，不需要生成中间列，避免后续 drop 操作
        rule_evaluations = [
            F.when(condition, F.lit(rule_name)).otherwise(None)
            for rule_name, condition in self.BASE_RULES.items()
        ]

         
        count_bronze = bronze_df.count() 
        print(f"count of count_bronze is {bronze_df}")

        count_enriched = enriched_df.count() 
        print(f"count of count_enriched is {count_enriched}")

        #dq_plan = enriched_df.withColumn(
        #    "violated_rules", 
        #    F.array_remove(F.array(*rule_evaluations), None)
        #).withColumn(
        #    "is_valid", 
        #    F.size(F.col("violated_rules")) == 0
        #)

        dq_plan = enriched_df.withColumn(
            "violated_rules", 
            # 确保即使 rule_evaluations 全是 None，也返回一个空数组 [] 而不是 NULL
            F.coalesce(F.array_remove(F.array(*rule_evaluations), None), F.array())
        ).withColumn(
            "is_valid", 
            # 增加 .getItem(0).isNull() 等防御性判断，或者直接确保结果是 boolean
            F.size(F.col("violated_rules")) == 0
        ).fillna({"is_valid": False})

        count_dq_plan = dq_plan.count() 
        print(f"count of count_dq_plan is {count_dq_plan}") 
        

        # --- 3. 缓存 DataFrame (极度重要) ---
        # 因为后续我们需要将数据分流（filter 两次），且在写入前需要执行 collect() 获取分区
        # 如果不 persist，上游的读取、特征计算和 DQ 校验会被触发重复计算 (DAG 重算)
        # dq_df.persist()

        print(">>> 正在统计 is_valid 分布...")
        # 这一步会强制触发计算并显示结果，解决你之前看不到 count 的问题
        dq_plan.groupBy("is_valid").count().show()
        
        print(">>> 数据样本展示 (前5行):")
        dq_plan.select("duration_min", "is_valid", "violated_rules").show(5, False) 

        temp_path = f"/Volumes/nyc/process_silver/checkpoint/{self.run_id}"

        (
            dq_plan
                .write
                .format("delta")
                .save(temp_path)
        )
        

        dq_df = spark.read.format("delta").load(temp_path)

        valid_count = dq_df.groupBy("is_valid").count()

        valid_count.show()


        try:
            # --- 4. 数据分流 (Data Splitting) ---
            valid_df = dq_df.filter(F.col("is_valid") == True).drop("violated_rules", "is_valid")
            rejected_df = dq_df.filter(F.col("is_valid") == False)

            count_valid = valid_df.count()
            count_rejected = rejected_df.count()

            print(f"Final Count - Valid: {count_valid}, Rejected: {count_rejected}")

            # --- 5. 幂等覆盖写入 ---
            if count_valid > 0: 
                self._write_to_delta(valid_df, self.target_table)
            if count_rejected > 0:
                self._write_to_delta(rejected_df, self.quarantine_table)
            
        finally:
            # 确保释放内存资源
            dbutils.fs.rm(temp_path, recurse=True)

    def _write_to_delta(self, df: DataFrame, table_name: str) -> None:
        # 此时 df 已经被 persist，这里的 collect() 是极速的，只扫描内存中的数据
        # partitions_rows = df.select("YYYYMM").distinct().collect()
        partitions_rows = [
            row["YYYYMM"]
            for row in (
                df.select("YYYYMM")
                .distinct()
                .toLocalIterator()
            )
        ]
        
        if not partitions_rows:
            print(f"No data to write for {table_name}. Skipping.")
            return

        partitions = [str(p) for p in partitions_rows]
        replace_cond = (
            f"YYYYMM IN ({','.join(partitions)})"
        )
        
        # 使用 Spark 3.x / Delta Lake 的原生动态分区覆盖也是一种选择
        # 但在生产环境中，显式指定 replaceWhere 更加安全，能防止意料之外的全局覆盖
        (
            df.write 
                .format("delta") 
                .mode("overwrite") 
                .option("replaceWhere", replace_cond) 
                .saveAsTable(table_name)
        )
        
        
        
        # 优化：去掉了原版的 df.count()，因为它会再次触发 Action
        print(f"Successfully loaded data to {table_name} for partitions {partitions}")

In [0]:
%sql
select * from nyc.process_bronze.brz_yellow_nyc_taxi limit 10; 

In [0]:
from pyspark.sql import SparkSession
from datetime import datetime
import uuid

# 假设这是你的 Databricks Notebook 或 PySpark 提交脚本的入口
spark = SparkSession.builder.appName("NYCTaxi_Silver_Processing").getOrCreate()

# 1. 定义你的环境参数
BRONZE_TABLE = "nyc.process_bronze.brz_yellow_nyc_taxi"

TARGET_YYYYMM = "201001" # 目标处理月份
# 实际生产中通常由调度工具(如 Airflow/Databricks Workflow)传入
RUN_ID = f"nyc_yellow_silver_run_{TARGET_YYYYMM}_{datetime.utcnow():%Y%m%d_%H%M%S}_{uuid.uuid4().hex[:8]}"

TARGET_TABLE = "nyc.process_silver.silver_yellow_taxi"

# 2. 实例化我们刚才优化的 Loader 类
silver_loader = NYC_Taxi_Silver_Loader(
    spark=spark, 
    run_id=RUN_ID, 
    target_table=TARGET_TABLE
)

# 3. 读取 Bronze 层数据 (按需过滤)
# 最佳实践：不要全表读取（spark.read.table），一定要带上下推过滤(Predicate Pushdown)
bronze_df = spark.table(BRONZE_TABLE).filter(f"YYYYMM = {TARGET_YYYYMM}")


# 4. 执行处理流程
silver_loader.process(bronze_df)

print(f"Pipeline finished successfully for run_id: {RUN_ID}")